**Silver to Gold: Building BI Ready Tables**

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType, IntegerType, FloatType, TimestampType

In [0]:
catalog_name = 'ecommerce'

In [0]:
df_gold_order_items = spark.table(f'{catalog_name}.silver.slv_order_items')
df_gold_order_items.limit(100).display()



In [0]:
# 1) Add gross amount
df_gold_order_items = df_gold_order_items.withColumn(
    "gross_amount",
    F.col("unit_price") * F.col("quantity")
)

# 2) Add discount_amount (discount_pct is already numeric, e.g. 21 -> 0.21%)
df_gold_order_items = df_gold_order_items.withColumn(
    "discount_amount",
    F.col("gross_amount") * (F.col("discount_pct") / 100.0)
)

# 3) Add net_amount = gross - discount
df_gold_order_items = df_gold_order_items.withColumn(
    "sale_amount",
    F.col("gross_amount") - F.col("discount_amount") + F.col("tax_amount")
)

# Add date id
df_gold_order_items = df_gold_order_items.withColumn(
    "date_id",
    F.date_format(F.col("dt"), "yyyyMMdd").cast(IntegerType()) # Create date_key
)

# Coupon flag
# coupon flag = 1 if coupon_code is not null, else 0
df_gold_order_items = df_gold_order_items.withColumn(
    "coupon_flag",
    F.when(F.col("coupon_code").isNotNull(), F.lit(1)).otherwise(F.lit(0))
)

df_gold_order_items.limit(100).display()



In [0]:
from datetime import datetime

# Generate the dynamic date string (Format: YYYY-MM-DD)
current_date = datetime.now().strftime("%Y-%m-%d")

# Create your dynamic Power BI reference header
pbi_note = f"# 1) ---Define your live FX rates (as of {current_date}, like your PBI note) ---"

print(pbi_note)


In [0]:

pbi_note = f"Live FX rates (as of {current_date}, like your PBI note) ---"
print(pbi_note)
# 1) ---Define your fixed FX rates (as of 2025-10-15, like your PBI note) ---
fx_rates = {
    "INR": 1.00,
    "AED": 24.18,
    "AUD": 57.55,
    "CAD": 62.98, 
    "GBP": 117.98,
    "SGD": 68.18,
    "USD": 88.29
}

rates = [(k, float(v)) for k, v in fx_rates.items()]
rates_df_gold_order_items = spark.createDataFrame(rates, ["currency", "inr_rate"])
rates_df_gold_order_items.show()

In [0]:
import requests
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

from datetime import datetime

# Generate the dynamic date string (Format: YYYY-MM-DD)
current_date = datetime.now().strftime("%Y-%m-%d")


# Initialize Spark Session (if not already running)
spark = SparkSession.builder.appName("LiveInrFXRates").getOrCreate()

# 1. Fetch live current FX rates using INR as the base currency
# (Using the free, keyless Frankfurter API)
url = "https://api.frankfurter.app/latest?base=INR"

try:
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    data = response.json()
except requests.exceptions.RequestException as e:
    raise SystemExit(f"Live API Connection Failed: {e}")

# 2. Extract rates from the API response
rates_dict = data.get("rates", {})

# 3. Format the data to match your exact structure
# The API returns how much foreign currency equals 1 INR (e.g., 1 INR = 0.009 GBP).
# To match your code (where 1 GBP = ~117 INR), we calculate the inverse: 1 / rate.
live_fx_rates = {"INR": 1.00}
for currency, rate in rates_dict.items():
    live_fx_rates[currency] = round(1.0 / float(rate), 2)

# 4. Filter for your specific required currencies
target_currencies = ["INR", "AED", "AUD", "CAD", "GBP", "SGD", "USD"]
filtered_rates = [
    (curr, float(live_fx_rates[curr])) 
    for curr in target_currencies 
    if curr in live_fx_rates
]

# 5. Define explicit schema for reliability
schema = StructType([
    StructField("currency", StringType(), False),
    StructField("inr_rate", DoubleType(), False)
])

# 6. Create your live DataFrame
rates_df_gold_order_items = spark.createDataFrame(filtered_rates, schema=schema)

# Create your dynamic Power BI reference header
pbi_note = f"---Live FX rates (as of {current_date}, like your PBI note) ---"

print(pbi_note)

# 7. Display the live results
rates_df_gold_order_items.show()


In [0]:
df_gold_order_items = (
    df_gold_order_items
    .join(
        rates_df_gold_order_items,
        rates_df_gold_order_items.currency == F.upper(F.trim(F.col("unit_price_currency"))),
        "left")
    .withColumn("sale_amount_inr", F.col("sale_amount")* F.col("inr_rate"))
    .withColumn("sale_amount_inr", F.ceil(F.col("sale_amount_inr")))
)

df_gold_order_items.limit(5).display()

In [0]:
df_gold_order_items = df_gold_order_items.select(
    F.col("date_id"),
    F.col("dt").alias("transaction_date"),
    F.col("order_ts").alias("transaction_ts"),
    F.col("customer_id"),
    F.col("order_id").alias("transaction_id5"),
    F.col("item_seq").alias("seq_no"),
    F.col("product_id"),
    F.col("quantity"),
    F.col("unit_price_currency"),
    F.col("unit_price"),
    F.col("discount_pct").alias("discount_percent"),
    F.col("tax_amount"),
    F.col("channel"),
    F.col("coupon_code"),
    F.col("gross_amount"),
    F.col("discount_amount"),
    F.col("sale_amount").alias("net_amount"),
    F.col("coupon_flag"),
    F.col("currency"),
    F.col("inr_rate"),
    F.col("sale_amount_inr").alias("net_amount_inr"),
    F.col("file_name"),
    F.col("ingest_timestamp")
)
df_gold_order_items.limit(100).display()


In [0]:
# Write raw data to the gold layer (catalog: ecomm, schema: gold, table: order_items)
df_gold_order_items.write.format("delta") \
.mode("overwrite") \
.option("overwriteSchema", "true") \
.saveAsTable(f"{catalog_name}.gold.gld_fact_order_items")

In [0]:
# Get record count
spark.sql(f"SELECT COUNT(*) FROM {catalog_name}.gold.gld_fact_order_items").show()

